#IMPORT DATA


In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go

## Load the CSV file into a DataFrame

The file has headers in the first row. `pandas.read_csv()` reads them automatically and creates a `DataFrame` from the data.

In [41]:


csv_candidates = (
    Path.cwd() / "src" / "metal_bands.csv",
    Path.cwd().parent / "src" / "metal_bands.csv",
)
csv_path = next((path for path in csv_candidates if path.is_file()), None)

if csv_path is None:
    raise FileNotFoundError("Could not find src/metal_bands.csv.")

df = pd.read_csv(csv_path, dtype={"Band ID": "string"})

print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns.")
print("Headers:", df.columns.tolist())
display(df.head())

Loaded 183397 rows and 7 columns.
Headers: ['Band ID', 'Name', 'URL', 'Country', 'Genre', 'Status', 'Photo_URL']


,Band ID,Name,URL,Country,Genre,Status,Photo_URL
0,3540442600,A // Solution,https://www.metal-archives.com/bands/A_--_Solu...,United States,Crust Punk/Thrash Metal,Split-up,NaN
1,3540525193,A Black Cold Diamond,https://www.metal-archives.com/bands/A_Black_C...,Colombia,Heavy/Doom Metal,Active,https://www.metal-archives.com/images/3/5/4/0/...
2,3540473101,A Billion Limbs,https://www.metal-archives.com/bands/A_Billion...,United Kingdom,Groove Metal/Deathcore,Active,NaN
3,3540535978,A Band Named Jon,https://www.metal-archives.com/bands/A_Band_Na...,United States,Brutal Death Metal/Grindcore,Active,https://www.metal-archives.com/images/3/5/4/0/...
4,3540352307,A Balance of Power,https://www.metal-archives.com/bands/A_Balance...,United States,Melodic Death Metal/Metalcore,Active,https://www.metal-archives.com/images/3/5/4/0/...


#ADDING COUNTER

In [42]:
df["COUNTER"]=1


#HEAVY METAL GENRES

# Keep only the first genre component before '/' or ';'.

In [43]:
# Keep only the columns needed for genre analysis.
df = df[["COUNTER", "Country", "Genre"]].copy()
# Keep the first genre component, then standardize genre labels.
df["Genre"] = df["Genre"].str.split("/", n=1).str[0].str.split(";", n=1).str[0].str.strip().str.upper()
# Expand shorthand genre labels before calculating totals.
df.loc[df["Genre"].eq("DEATH"), "Genre"] = "DEATH METAL"
df.loc[df["Genre"].eq("THRASH"), "Genre"] = "THRASH METAL"
df.loc[df["Genre"].eq("HEAVY"), "Genre"] = "HEAVY METAL"
df.loc[df["Genre"].eq("BLACK"), "Genre"] = "BLACK METAL"
df.loc[df["Genre"].eq("STONER"), "Genre"] = "STONER METAL"
df.loc[df["Genre"].eq("BLACK"), "Genre"] = "BLACK METAL"
df.loc[df["Genre"].eq("SLUDGE"), "Genre"] = "SLUDGE METAL"
df.loc[df["Genre"].eq("DOOM"), "Genre"] = "DOOM METAL"
df.loc[df["Genre"].eq("MELODIC DEATH"), "Genre"] = "MELODIC DEATH METAL"



# Aggregate band counts by country and genre, and by genre alone.
country_genre_totals = df.groupby(["Country", "Genre"], as_index=False)["COUNTER"].sum()
genre_totals = df.groupby("Genre", as_index=False)["COUNTER"].sum()

# Map each country label to its continent.
country_to_continent = {
    **dict.fromkeys(["Algeria", "Angola", "Botswana", "Egypt", "Ethiopia", "Kenya", "Libya", "Madagascar", "Malawi", "Mauritius", "Morocco", "Mozambique", "Namibia", "Reunion", "South Africa", "Tunisia", "Uganda", "Zimbabwe"], "Africa"),
    **dict.fromkeys(["Afghanistan", "Armenia", "Azerbaijan", "Bahrain", "Bangladesh", "Brunei", "Cambodia", "China", "Cyprus", "East Timor", "Georgia", "Hong Kong", "India", "Indonesia", "Iran", "Iraq", "Israel", "Japan", "Jordan", "Kazakhstan", "Korea, South", "Kuwait", "Kyrgyzstan", "Laos", "Lebanon", "Malaysia", "Maldives", "Mongolia", "Myanmar", "Nepal", "Oman", "Pakistan", "Palestine", "Philippines", "Qatar", "Saudi Arabia", "Singapore", "Sri Lanka", "Syria", "Taiwan", "Tajikistan", "Thailand", "Turkmenistan", "T\u00fcrkiye", "United Arab Emirates", "Uzbekistan", "Vietnam"], "Asia"),
    **dict.fromkeys(["Albania", "Andorra", "Austria", "\u00c5land Islands", "Belarus", "Belgium", "Bosnia and Herzegovina", "Bulgaria", "Croatia", "Czechia", "Denmark", "Estonia", "Faroe Islands", "Finland", "France", "Germany", "Gibraltar", "Greece", "Guernsey", "Hungary", "Iceland", "Ireland", "Isle of Man", "Italy", "Jersey", "Latvia", "Liechtenstein", "Lithuania", "Luxembourg", "Malta", "Moldova", "Monaco", "Montenegro", "Netherlands", "North Macedonia", "Norway", "Poland", "Portugal", "Romania", "Russia", "San Marino", "Serbia", "Slovakia", "Slovenia", "Svalbard", "Spain", "Sweden", "Switzerland", "Ukraine", "United Kingdom"], "Europe"),
    **dict.fromkeys(["Aruba", "Barbados", "Belize", "Canada", "Costa Rica", "Cuba", "Cura\u00e7ao", "Dominican Republic", "El Salvador", "Greenland", "Guadeloupe", "Guatemala", "Honduras", "Jamaica", "Mexico", "Nicaragua", "Panama", "Puerto Rico", "Saint Pierre and Miquelon", "Trinidad and Tobago", "United States"], "North America"),
    **dict.fromkeys(["Argentina", "Bolivia", "Brazil", "Chile", "Colombia", "Ecuador", "Falkland Islands", "French Guiana", "Guyana", "Paraguay", "Peru", "Suriname", "Uruguay", "Venezuela"], "South America"),
    **dict.fromkeys(["Australia", "French Polynesia", "Guam", "New Caledonia", "New Zealand"], "Oceania"),
    "International": "Multiple continents",
    "Unknown": "Unknown",
}

unmapped_countries = sorted(set(df["Country"]) - country_to_continent.keys())
if unmapped_countries:
    raise ValueError(f"No continent mapping for countries: {', '.join(unmapped_countries)}")

bands_with_continent = df[["Country", "Genre", "COUNTER"]].copy()
bands_with_continent["Continent"] = bands_with_continent["Country"].map(country_to_continent)

display(df.head())
display(country_genre_totals.head())
display(genre_totals.head())
display(bands_with_continent.head())

,COUNTER,Country,Genre
0,1,United States,CRUST PUNK
1,1,Colombia,HEAVY METAL
2,1,United Kingdom,GROOVE METAL
3,1,United States,BRUTAL DEATH METAL
4,1,United States,MELODIC DEATH METAL


,Country,Genre,COUNTER
0,Afghanistan,BLACK METAL,1
1,Afghanistan,MELODIC DEATH METAL,1
2,Afghanistan,PROGRESSIVE GROOVE METAL,1
3,Albania,BLACK METAL,4
4,Albania,DEATH METAL,3


,Genre,COUNTER
0,A CAPPELLA,1
1,ACOUSTIC,3
2,ACOUSTIC (EARLY),1
3,ACOUSTIC DEATH,1
4,ACOUSTIC ROCK (EARLY),2


,Country,Genre,COUNTER,Continent
0,United States,CRUST PUNK,1,North America
1,Colombia,HEAVY METAL,1,South America
2,United Kingdom,GROOVE METAL,1,Europe
3,United States,BRUTAL DEATH METAL,1,North America
4,United States,MELODIC DEATH METAL,1,North America


#CREATING VISUALIZATION 

In [44]:
# Keep labeled genres with positive counts above the existing cutoff.
genre_totals_filtered = genre_totals.loc[
    genre_totals["Genre"].notna()
    & genre_totals["Genre"].str.strip().ne("")
    & genre_totals["COUNTER"].notna()
    & genre_totals["COUNTER"].gt(2000)
].copy()
print(genre_totals_filtered)

                    Genre  COUNTER
182           BLACK METAL    31677
374    BRUTAL DEATH METAL     4238
439           DEATH METAL    26653
537            DOOM METAL     4220
921           HEAVY METAL    20869
1132  MELODIC DEATH METAL     8118
1263            METALCORE     2090
1414          POWER METAL     2116
1506    PROGRESSIVE METAL     3709
1670      RAW BLACK METAL     2767
1724         SLUDGE METAL     3253
1752         STONER METAL     3206
1927         THRASH METAL    18770


In [45]:
# Plot only genres with non-empty labels and positive counts.
genre_plot_data = genre_totals_filtered.loc[
    genre_totals_filtered["Genre"].notna()
    & genre_totals_filtered["Genre"].str.strip().ne("")
    & genre_totals_filtered["COUNTER"].notna()
    & genre_totals_filtered["COUNTER"].gt(0)
].sort_values("COUNTER", ascending=True)

genre_fig = go.Figure(go.Bar(
    x=genre_plot_data["COUNTER"],
    y=genre_plot_data["Genre"],
    orientation="h",
    marker=dict(
        color=genre_plot_data["COUNTER"],
        colorscale=[[0, "#451019"], [0.5, "#922537"], [1, "#E06A78"]],
        line=dict(color="#F0B5BE", width=0.8),
    ),
    text=[f"{count:,}" for count in genre_plot_data["COUNTER"]],
    textposition="outside",
    textfont=dict(color="#F0E7E9", size=12),
    hovertemplate="<b>%{y}</b><br>Number of bands: %{x:,}<extra></extra>",
))

genre_fig.update_layout(
    title=dict(text="Most Popular Metal Genres", font=dict(size=24, color="#F0B5BE")),
    template="plotly_dark",
    paper_bgcolor="#08080B",
    plot_bgcolor="#111116",
    font=dict(family="Arial", size=13, color="#E6DEE0"),
    height=680,
    margin=dict(l=190, r=65, t=80, b=55),
    xaxis=dict(
        title="Number of bands",
        range=[0, genre_plot_data["COUNTER"].max() * 1.15],
        showgrid=True,
        gridcolor="#332A30",
        zeroline=False,
    ),
    yaxis=dict(title=None, showgrid=False),
    showlegend=False,
)

genre_fig.show()

## TOP 10 GENRES BY CONTINENT

In [46]:
# Analyze geographic continents only; exclude non-geographic labels.
continents_for_analysis = [
    "Africa", "Asia", "Europe", "North America", "Oceania", "South America",
]

# Remove missing labels and non-positive counts before aggregation.
bands_for_continent_analysis = bands_with_continent.loc[
    bands_with_continent["Continent"].isin(continents_for_analysis)
    & bands_with_continent["Genre"].notna()
    & bands_with_continent["Genre"].str.strip().ne("")
    & bands_with_continent["COUNTER"].notna()
    & bands_with_continent["COUNTER"].gt(0)
]

# Sum band counts by continent and genre.
continent_genre_totals = (
    bands_for_continent_analysis
    .groupby(["Continent", "Genre"], as_index=False)["COUNTER"]
    .sum()
)

# Rank genres per continent and keep the ten highest counts.
top10_genres_by_continent = (
    continent_genre_totals
    .sort_values(
        ["Continent", "COUNTER", "Genre"],
        ascending=[True, False, True],
    )
    .groupby("Continent", sort=False, group_keys=False)
    .head(10)
    .reset_index(drop=True)
)

display(top10_genres_by_continent)

top10_chart_data = top10_genres_by_continent.loc[
    top10_genres_by_continent["Continent"].notna()
    & top10_genres_by_continent["Genre"].notna()
    & top10_genres_by_continent["Genre"].str.strip().ne("")
    & top10_genres_by_continent["COUNTER"].notna()
    & top10_genres_by_continent["COUNTER"].gt(0)
].sort_values(
    ["Continent", "COUNTER", "Genre"],
    ascending=[True, True, True],
)

# Draw a separate chart with its own genre legend for each continent.
metal_palette = [
    "#7B1728", "#A61F37", "#CC344D", "#E05B6D", "#5C111C",
    "#8F3140", "#B65360", "#D9848D", "#B99197", "#E4C0C5",
]
top10_continent_figures = {}

for continent in continents_for_analysis:
    continent_plot_data = top10_chart_data.loc[
        top10_chart_data["Continent"].eq(continent)
    ].sort_values(["COUNTER", "Genre"], ascending=[True, True])
    if continent_plot_data.empty:
        raise ValueError(f"No chart data available for {continent}.")

    continent_fig = px.bar(
        continent_plot_data,
        x="COUNTER",
        y="Genre",
        orientation="h",
        color="Genre",
        color_discrete_sequence=metal_palette,
        text="COUNTER",
        custom_data=["Continent"],
        category_orders={"Genre": continent_plot_data["Genre"].tolist()},
        labels={"COUNTER": "Number of bands", "Genre": "Genre"},
        title=f"Top 10 Metal Genres - {continent}",
    )
    continent_fig.update_traces(
        texttemplate="%{x:,}",
        textposition="outside",
        textfont=dict(color="#F0E7E9", size=11),
        cliponaxis=False,
        hovertemplate="<b>%{y}</b><br>Continent: %{customdata[0]}<br>Number of bands: %{x:,}<extra></extra>",
    )
    continent_fig.update_layout(
        template="plotly_dark",
        paper_bgcolor="#08080B",
        plot_bgcolor="#111116",
        font=dict(family="Arial", size=12, color="#E6DEE0"),
        title=dict(font=dict(size=22, color="#F0B5BE")),
        width=1100,
        height=600,
        margin=dict(l=190, r=260, t=80, b=55),
        legend=dict(
            title="Genre",
            x=1.02,
            y=1,
            xanchor="left",
            yanchor="top",
            bgcolor="#111116",
            bordercolor="#5A3038",
            borderwidth=1,
        ),
    )
    continent_fig.update_xaxes(
        title="Number of bands",
        range=[0, continent_plot_data["COUNTER"].max() * 1.15],
        showgrid=True,
        gridcolor="#332A30",
        zeroline=False,
    )
    continent_fig.update_yaxes(showgrid=False, title=None)
    top10_continent_figures[continent] = continent_fig
    continent_fig.show()

,Continent,Genre,COUNTER
0,Africa,DEATH METAL,95
1,Africa,BLACK METAL,87
2,Africa,THRASH METAL,46
3,Africa,HEAVY METAL,43
4,Africa,MELODIC DEATH METAL,43
5,Africa,PROGRESSIVE METAL,21
6,Africa,GROOVE METAL,14
7,Africa,GROOVE,11
8,Africa,DOOM METAL,9
9,Africa,MELODIC BLACK,9


In [ ]:
# Export the figures from this analysis as reusable PNG files.
visualization_dir = csv_path.parent.parent / "visualization"
visualization_dir.mkdir(parents=True, exist_ok=True)

figures_to_export = {
    "most_popular_metal_genres": genre_fig,
    **{
        f"top_10_metal_genres_{continent.lower().replace(' ', '_')}": figure
        for continent, figure in top10_continent_figures.items()
    },
}

for filename, figure in figures_to_export.items():
    output_path = visualization_dir / f"{filename}.png"
    width = figure.layout.width or 700
    height = figure.layout.height or 500
    figure.write_image(str(output_path), width=width, height=height, scale=2)
    print(f"Saved {output_path}")